# EBRT Monte Carlo Dataset Consolidation — Varian TrueBeam SVC 3.0

## Management notebook: N datasets → 1 unified dataset

---

### Purpose

Monte Carlo simulations run across **multiple Kaggle notebooks** (distributed across several accounts). Each notebook produces:
- An `ebrt_batch/` directory with per-scenario subdirectories (`.mhd` + `_metadata.json` files)
- A partial HDF5 `ebrt_training_dataset.h5` via `prepare_ebrt_dataset()`

This notebook **consolidates everything** into a single `ebrt_consolidated_dataset.h5` ready for neural network training.

### Operation mode: HDF5 + reconstruction from raw files

Partial HDF5 files from Kaggle often become **corrupted** because the notebook gets cut off by time or disk limits before closing the file (`"bad object header version number"`). Therefore this notebook has **two discovery modes**:

1. **Valid HDF5**: If found, they are copied directly (faster)
2. **Raw files** (`.mhd` + `_metadata.json`): If HDF5 files are corrupt or missing, the dataset is **reconstructed** from raw simulation files using the same functions as `prepare_ebrt_dataset()` from the simulation notebook

### "Duplicate" samples? → They are NOT duplicates

Multiple simulations with the **same parameters** but **different random seeds** produce **different** dose maps (Monte Carlo is stochastic). ALL are kept because:

1. **Noise reduction**: $N$ independent realizations ≡ $N\times$ more particles
2. **Natural data augmentation** for the neural network
3. **Robustness**: the network learns distributions, not single values

### HDF5 dataset structure (10 parameters)

| Dataset | Shape | Content |
|---|---|---|
| `/density` | `(N, 1, 200, 200, 200)` | Normalized density maps (÷1.85) |
| `/params` | `(N, 10)` | Normalized beam parameters [0, 1] |
| `/dose` | `(N, 1, 200, 200, 200)` | MC dose maps (Gy) |

> **Note**: MC uncertainty (σ_D/D) is **not stored**. Detailed analysis concluded that dose-weighted MSE already approximates the optimal 1/σ² weighting, the storage savings are ~30%, and MC Dropout is superior for estimating epistemic uncertainty in predictions. Mean uncertainty is computed and logged as a quality metric during consolidation.

| # | Parameter | Real range | Normalization |
|---|---|---|---|
| 0 | `beam_energy_norm` | 6/10/15 MV | 0.0=6MV, 0.5=10MV, 1.0=15MV |
| 1 | `is_fff` | 0/1 | 0=FF, 1=FFF |
| 2 | `field_x_cm` | 1–40 cm | (x-1)/39 |
| 3 | `field_y_cm` | 1–22 cm | (x-1)/21 |
| 4 | `ssd_cm` | 80–110 cm | (x-80)/30 |
| 5 | `gantry_angle` | 0–360° | x/360 |
| 6 | `collimator_angle` | 0–360° | x/360 |
| 7 | `patient_type` | pelvis/head/lung | 0=pelvis, 0.5=head_neck, 1.0=lung |
| 8 | `tumor_depth` | 2–18 cm | (x-2)/16 |
| 9 | `tumor_size` | 1–8 cm | (x-1)/7 |

### Usage instructions on Kaggle

1. Simulation notebooks save their output as a **Kaggle Dataset**
2. In this notebook, add all those datasets as **Input** (`/kaggle/input/{name}/`)
3. Run all cells → generates `ebrt_consolidated_dataset.h5` in `/kaggle/working/`
4. Save this output as a new Kaggle Dataset
5. In `red_neuronal_EBRT.ipynb`, add that consolidated dataset as Input

In [ ]:
    # =============================================================================
    # CELL 1: Imports, configuration and MHD reading functions
    # =============================================================================

    import os
    import json
    import numpy as np
    import h5py
    from collections import defaultdict
    import time
    import shutil
    import subprocess
    import re as _re

    # ─── Detect environment ───
    ON_KAGGLE = os.path.exists('/kaggle/working')

    if ON_KAGGLE:
        INPUT_ROOT = '/kaggle/input'
        OUTPUT_DIR = '/kaggle/working'
    else:
        INPUT_ROOT = os.path.join(os.getcwd(), 'output_ebrt')
        OUTPUT_DIR = os.path.join(os.getcwd(), 'output_ebrt')

    OUTPUT_H5 = os.path.join(OUTPUT_DIR, 'ebrt_consolidated_dataset.h5')
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # Name expected by red_neuronal_EBRT.ipynb (_find_dataset_h5)
    # Priority 0: ebrt_consolidated_dataset.h5,  Priority 1: ebrt_training_dataset.h5
    EXPECTED_H5_NAMES = ['ebrt_consolidated_dataset.h5', 'ebrt_training_dataset.h5']

    # ─── Batch upload: upload HDF5 to Kaggle and free disk to continue ───
    # Allows processing ALL samples in ONE single run
    # (without this, each run is limited to ~810 samples ≈ 19 GB of disk)
    BATCH_UPLOAD_ENABLED = ON_KAGGLE  # Auto-activar en Kaggle
    KAGGLE_USERNAME = os.environ.get('KAGGLE_USERNAME', '') or 'ibongarciagomez'

    # ─── 10 EBRT Parameters — consistent with prepare_ebrt_dataset() ───
    EBRT_PARAM_NAMES = [
        'beam_energy_norm', 'is_fff', 'field_x_cm', 'field_y_cm', 'ssd_cm',
        'gantry_angle', 'collimator_angle', 'patient_type', 'tumor_depth', 'tumor_size',
    ]
    N_PARAMS_EXPECTED = 10

    EBRT_PARAM_RANGES = {
        'beam_energy_norm': (0.0, 1.0),
        'is_fff':           (0.0, 1.0),
        'field_x_cm':       (1.0, 40.0),
        'field_y_cm':       (1.0, 22.0),
        'ssd_cm':           (80.0, 110.0),
        'gantry_angle':     (0.0, 360.0),
        'collimator_angle': (0.0, 360.0),
        'patient_type':     (0.0, 1.0),
        'tumor_depth':      (2.0, 18.0),
        'tumor_size':       (1.0, 8.0),
    }

    # Beam quality mappings (identical to prepare_ebrt_dataset)
    ENERGY_MAP = {'6MV': 0.0, '6FFF': 0.0, '10MV': 0.5, '10FFF': 0.5, '15MV': 1.0}
    FFF_MAP    = {'6MV': 0.0, '6FFF': 1.0, '10MV': 0.0, '10FFF': 1.0, '15MV': 0.0}
    PATIENT_TYPE_MAP = {'pelvis': 0.0, 'head_neck': 0.5, 'lung': 1.0}

    # Expected volume: 40×40×40 cm with spacing 2mm → (200, 200, 200)
    EXPECTED_VOL_SHAPE = (200, 200, 200)

    print(f"{'Kaggle' if ON_KAGGLE else 'Local'}")
    print(f"  Input root:  {INPUT_ROOT}")
    print(f"  Output HDF5: {OUTPUT_H5}")
    print(f"  Parameters:  {N_PARAMS_EXPECTED}")
    print(f"  Volumen:     {EXPECTED_VOL_SHAPE}")
    print(f"  Batch upload: {'ENABLED' if BATCH_UPLOAD_ENABLED else 'disabled'}")
    if BATCH_UPLOAD_ENABLED:
        print(f"  Kaggle user:  {KAGGLE_USERNAME or '(not detected)'}")

    try:
        usage = shutil.disk_usage(OUTPUT_DIR)
        print(f"  Free disk: {usage.free / 1e9:.1f} GB")
    except Exception:
        pass


    # =============================================================================
    # MHD reading functions (copied from ebrt_montecarlo_simulacion.ipynb)
    # =============================================================================

    def load_mhd_manual(mhd_path):
        """Loads an MHD file manually, without SimpleITK."""
        info = {}
        with open(mhd_path, 'r') as f:
            for line in f:
                if '=' in line:
                    k, v = line.split('=', 1)
                    info[k.strip()] = v.strip()
        dims = list(map(int, info['DimSize'].split()))
        spacing = list(map(float, info['ElementSpacing'].split()))
        offset = list(map(float, info.get('Offset', '0 0 0').split()))
        dtype_map = {
            'MET_FLOAT': np.float32, 'MET_DOUBLE': np.float64,
            'MET_SHORT': np.int16, 'MET_UCHAR': np.uint8,
        }
        dtype = dtype_map.get(info['ElementType'], np.float32)
        raw_file = os.path.join(os.path.dirname(mhd_path), info['ElementDataFile'])
        data = np.fromfile(raw_file, dtype=dtype).reshape(dims[2], dims[1], dims[0])
        return data, {'dims': dims, 'spacing': spacing, 'offset': offset}


    def load_mhd(mhd_path):
        """Loads an MHD file using SimpleITK (preferred) or manual reading."""
        try:
            import SimpleITK as sitk
            img = sitk.ReadImage(mhd_path)
            arr = sitk.GetArrayFromImage(img).astype(np.float64)
            meta = {
                'dims': list(img.GetSize()),
                'spacing': list(img.GetSpacing()),
                'offset': list(img.GetOrigin()),
            }
            return arr, meta
        except ImportError:
            return load_mhd_manual(mhd_path)


    # =============================================================================
    # Analytical density map generation (copied from simulation)
    # =============================================================================

    def build_ebrt_density(sim_meta, vol_shape, spacing_mm, offset_mm):
        """
        Builds an analytical 3D density map for an EBRT scenario.
        Exact replica of the function from the simulation notebook.
        """
        nz, ny, nx = vol_shape
        x = np.arange(nx, dtype=np.float32) * spacing_mm[0] + offset_mm[0]
        y = np.arange(ny, dtype=np.float32) * spacing_mm[1] + offset_mm[1]
        z = np.arange(nz, dtype=np.float32) * spacing_mm[2] + offset_mm[2]
        Z, Y, X = np.meshgrid(z, y, x, indexing='ij')

        density = np.ones(vol_shape, dtype=np.float32)
        geom = sim_meta.get('geometry', {})
        phantom_hz = geom.get('phantom_size_cm', [40.0, 40.0, 40.0])[2] / 2.0

        # Bone (1.85 g/cm³) — horizontal slab
        bd = geom.get('bone_depth_cm', 8.0)
        bt = geom.get('bone_thickness_cm', 0.5)
        bone_z = (phantom_hz - bd - bt / 2.0) * 10.0
        bone_mask = np.abs(Z - bone_z) <= (bt * 10.0 / 2.0)
        density[bone_mask] = 1.85

        # OAR (1.04 g/cm³) — sphere
        oar_pos = geom.get('oar_position_cm', [5.0, 0.0, 8.0])
        oar_r = geom.get('oar_radius_cm', 1.5) * 10.0
        oar_z = (phantom_hz - oar_pos[2]) * 10.0
        oar_mask = np.sqrt((X - oar_pos[0]*10)**2 + (Y - oar_pos[1]*10)**2 +
                        (Z - oar_z)**2) <= oar_r
        density[oar_mask] = 1.04

        # Lung (0.26 g/cm³) — rectangular box
        if geom.get('has_lung') and geom.get('lung_position_cm'):
            lp = geom['lung_position_cm']
            ls = geom['lung_size_cm']
            lung_z = (phantom_hz - lp[2]) * 10.0
            lung_mask = (np.abs(X - lp[0]*10) <= ls[0]*10/2) & \
                        (np.abs(Y - lp[1]*10) <= ls[1]*10/2) & \
                        (np.abs(Z - lung_z) <= ls[2]*10/2)
            density[lung_mask] = 0.26

        return density


    # =============================================================================
    # Batch upload: upload partial HDF5 to Kaggle and free disk
    # =============================================================================

    def _get_next_part_number():
        """Determines the next part number by searching existing inputs."""
        existing = set()
        for dirpath, dirnames, filenames in os.walk(INPUT_ROOT):
            for fname in filenames:
                m = _re.search(r'part[_-]?(\d+)', fname)
                if m:
                    existing.add(int(m.group(1)))
        # Also search in working in case there are files from previous runs in the same session
        for fname in os.listdir(OUTPUT_DIR):
            m = _re.search(r'part[_-]?(\d+)', fname)
            if m:
                existing.add(int(m.group(1)))
        return max(existing, default=0) + 1


    def _upload_kaggle_dataset(h5_path, part_number):
        """
        Uploads an HDF5 to Kaggle as an independent dataset and deletes it from disk.

        Kaggle working = ~20 GB. This allows:
        1. Write ~810 samples (19 GB)
        2. Upload to Kaggle → free 19 GB
        3. Repeat → process ALL samples in ONE run

        Returns True if the upload was successful.
        """
        if not KAGGLE_USERNAME:
            print("   ⚠ KAGGLE_USERNAME not detected — cannot upload")
            return False

        slug = f"{KAGGLE_USERNAME}/ebrt-consolidated-part-{part_number:03d}"
        title = f"EBRT Consolidated Part {part_number:03d}"

        # Create temporary directory for upload (avoid uploading _index.json)
        upload_dir = os.path.join(OUTPUT_DIR, f'_upload_tmp_{part_number}')
        os.makedirs(upload_dir, exist_ok=True)

        # Move HDF5 to upload directory
        h5_name = os.path.basename(h5_path)
        dest_path = os.path.join(upload_dir, h5_name)
        shutil.move(h5_path, dest_path)

        # Write dataset-metadata.json (required by Kaggle API)
        meta = {
            "title": title,
            "id": slug,
            "licenses": [{"name": "CC0-1.0"}]
        }
        with open(os.path.join(upload_dir, "dataset-metadata.json"), 'w') as f:
            json.dump(meta, f)

        file_size_gb = os.path.getsize(dest_path) / 1e9
        print(f"\n   {'='*55}")
        print(f"   BATCH UPLOAD: Part {part_number}")
        print(f"   File:     {h5_name} ({file_size_gb:.1f} GB)")
        print(f"   Destino:  {slug}")
        print(f"   {'='*55}")

        t_upload = time.time()

        # Try to create new dataset
        result = subprocess.run(
            ["kaggle", "datasets", "create", "-p", upload_dir],
            capture_output=True, text=True, timeout=7200
        )

        # If it already exists, try a new version
        if result.returncode != 0:
            result = subprocess.run(
                ["kaggle", "datasets", "version", "-p", upload_dir,
                "-m", f"Part {part_number} reconsolidated"],
                capture_output=True, text=True, timeout=7200
            )

        upload_time = time.time() - t_upload
        success = result.returncode == 0

        if success:
            print(f"   Upload OK ({upload_time:.0f}s)")
            # Clean upload directory → free disk
            shutil.rmtree(upload_dir, ignore_errors=True)
            freed = file_size_gb
            free_after = shutil.disk_usage(OUTPUT_DIR).free / 1e9
            print(f"   Freed {freed:.1f} GB → free disk: {free_after:.1f} GB")
        else:
            print(f"   Upload FAILED ({upload_time:.0f}s)")
            if result.stdout.strip():
                print(f"   stdout: {result.stdout.strip()[:200]}")
            if result.stderr.strip():
                print(f"   stderr: {result.stderr.strip()[:200]}")
            # Return the file to the working dir
            if os.path.exists(dest_path):
                shutil.move(dest_path, h5_path)
            shutil.rmtree(upload_dir, ignore_errors=True)
            print(f"   The file remains at {h5_path} (save it manually)")

        return success

    print("Functions loaded: load_mhd, build_ebrt_density, _upload_kaggle_dataset")

In [ ]:
# =============================================================================
# CELL 2: Auto-discovery of data — valid HDF5 + raw files (.mhd)
# =============================================================================
# Kaggle HDF5 files frequently become corrupted ("bad object header version
# number") because the notebook gets cut off by time or disk.
#
# Therefore we search for TWO types of sources:
#   1. HDF5 with 10 params, density, params, dose → direct reading (fast)
#   2. Directorios ebrt_batch/ con {sid}_dose.mhd + {sid}_metadata.json
#      → reconstruction from raw files (slower, but rescues data)
#
# Also detects previous consolidated files for incremental consolidation.
#
# RESUME: If an _index.json from a previous run exists (in Kaggle
# inputs), it loads it to know which SIDs are already consolidated and
# not repeat them.
# =============================================================================

def discover_raw_simulations(root_dir):
    """
    Recursively searches for ebrt_batch-type directories containing
    raw simulations ({sid}_dose.mhd + {sid}_metadata.json).

    Returns: list of dicts with info about each valid simulation found
    """
    raw_sims = []
    batch_dirs_found = set()

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for fname in filenames:
            if not fname.endswith('_metadata.json'):
                continue

            sid = fname.replace('_metadata.json', '')
            meta_path = os.path.join(dirpath, fname)
            dose_path = os.path.join(dirpath, f'{sid}_dose.mhd')
            dose_raw_path = os.path.join(dirpath, f'{sid}_dose.raw')

            if not os.path.exists(dose_path):
                continue
            if not os.path.exists(dose_raw_path):
                continue

            try:
                with open(meta_path) as f:
                    meta = json.load(f)
                if 'error' in meta:
                    continue
            except Exception:
                continue

            batch_dir = os.path.dirname(dirpath)
            batch_dirs_found.add(batch_dir)

            uncert_path = os.path.join(dirpath, f'{sid}_dose_uncertainty.mhd')
            has_uncert = os.path.exists(uncert_path) and \
                         os.path.exists(os.path.join(dirpath, f'{sid}_dose_uncertainty.raw'))

            raw_sims.append({
                'sid': sid,
                'sim_dir': dirpath,
                'meta_path': meta_path,
                'dose_path': dose_path,
                'uncert_path': uncert_path if has_uncert else None,
                'has_uncertainty': has_uncert,
                'metadata': meta,
                'batch_dir': batch_dir,
            })

    return raw_sims, batch_dirs_found


def discover_h5_files(root_dir):
    """
    Recursively searches for valid HDF5 files with EBRT structure (10 params).
    Attempts to open them — if they are corrupt, it reports and skips them.
    """
    consolidated_prev = []
    partials = []
    corrupted = []

    if not os.path.isdir(root_dir):
        return consolidated_prev, partials, corrupted

    h5_candidates = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname.endswith('.h5') or fname.endswith('.hdf5'):
                h5_candidates.append(os.path.join(dirpath, fname))

    for h5_path in sorted(h5_candidates):
        rel_path = os.path.relpath(h5_path, root_dir)
        try:
            with h5py.File(h5_path, 'r') as hf:
                required_fields = {'density', 'params', 'dose'}
                if not required_fields.issubset(set(hf.keys())):
                    continue

                n = hf['density'].shape[0]
                if n == 0:
                    continue

                vol_shape = tuple(hf['density'].shape[2:])
                n_params = hf['params'].shape[1]
                has_uncert = 'uncertainty' in hf
                is_consolidated = bool(hf.attrs.get('consolidated', False))

                if n_params != N_PARAMS_EXPECTED:
                    print(f"  [HDF5] Skipping {rel_path}: {n_params} params "
                          f"(not EBRT 10-param)")
                    continue

                dose_max = float(hf.attrs.get('global_dose_max',
                                 hf.attrs.get('dose_max_global', 0)))
                source = rel_path.split(os.sep)[0] if os.sep in rel_path else rel_path

                info = {
                    'path': h5_path, 'n_samples': n, 'volume_shape': vol_shape,
                    'n_params': n_params, 'has_uncertainty': has_uncert,
                    'dose_max_global': dose_max, 'source': source,
                    'is_consolidated': is_consolidated,
                }

                if is_consolidated:
                    consolidated_prev.append(info)
                    is_partial = bool(hf.attrs.get('is_partial', False))
                    tag = " (PARTIAL)" if is_partial else ""
                    print(f"  [HDF5] PREVIOUS CONSOLIDATED{tag}: {rel_path}: "
                          f"{n} samples, vol={vol_shape}")
                else:
                    partials.append(info)
                    print(f"  [HDF5] Partial: {rel_path}: {n} samples, "
                          f"vol={vol_shape}")

        except Exception as e:
            error_msg = str(e)
            corrupted.append({'path': h5_path, 'error': error_msg})
            print(f"  [HDF5] CORRUPTO: {rel_path}: {error_msg[:80]}")

    return consolidated_prev, partials, corrupted


def discover_resume_index(root_dir):
    """
    Searches for _index.json files from previous consolidations.
    Required for incremental resume: to know which SIDs are already consolidated.
    """
    index_files = []
    for dirpath, _, filenames in os.walk(root_dir):
        for fname in filenames:
            if fname == 'ebrt_consolidated_dataset_index.json':
                index_files.append(os.path.join(dirpath, fname))

    if index_files:
        # Use the most recent one (the one with the most SIDs)
        best = None
        best_count = 0
        for idx_path in index_files:
            try:
                with open(idx_path) as f:
                    data = json.load(f)
                count = data.get('n_consolidated', 0)
                if count > best_count:
                    best = idx_path
                    best_count = count
            except Exception:
                pass
        if best:
            print(f"  [INDEX] Resume found: {best_count} SIDs already consolidated")
            # Copy to working directory so merge_all_sources can find it
            dest = os.path.join(OUTPUT_DIR, 'ebrt_consolidated_dataset_index.json')
            if not os.path.exists(dest):
                shutil.copy2(best, dest)
                print(f"  [INDEX] Copiado a {dest}")
            return best
    return None


# ─── Run discovery ───
print(f"Searching for EBRT data in {INPUT_ROOT}...\n")
print("── Step 1: HDF5 ──")
consolidated_prev, h5_partials, h5_corrupted = discover_h5_files(INPUT_ROOT)

print(f"\n── Step 2: Resume index ──")
resume_index = discover_resume_index(INPUT_ROOT)

print(f"\n── Step 3: Raw simulations (.mhd + metadata.json) ──")
raw_sims, batch_dirs = discover_raw_simulations(INPUT_ROOT)

# ─── Summary ───
n_h5_valid = sum(d['n_samples'] for d in h5_partials)
n_h5_prev = sum(d['n_samples'] for d in consolidated_prev)
n_raw = len(raw_sims)
n_corrupted = len(h5_corrupted)

print(f"\nDiscovery summary:")
if consolidated_prev:
    print(f"   Previous consolidated (HDF5): {len(consolidated_prev)} "
          f"({n_h5_prev} samples)")
if h5_partials:
    print(f"   Valid partial HDF5:           {len(h5_partials)} "
          f"({n_h5_valid} samples)")
if n_corrupted > 0:
    print(f"   HDF5 CORRUPTOS:              {n_corrupted} "
          f"(will be reconstructed from raw)")
if raw_sims:
    print(f"   Raw simulations (.mhd):       {n_raw} "
          f"(in {len(batch_dirs)} batch directories)")
print(f"   Total estimado:              "
      f"{n_h5_prev + n_h5_valid + n_raw} samples")

if not consolidated_prev and not h5_partials and not raw_sims:
    print("\nNo valid EBRT data found.")
    print("   Add the simulation notebook outputs as input datasets.")
    print("   Both HDF5 and raw files (.mhd + _metadata.json) are searched.")
else:
    if n_corrupted > 0 and n_raw > 0:
        print(f"\n   The {n_corrupted} corrupted HDF5 will be replaced by "
              f"reconstructed raw simulations.")

In [ ]:
# =============================================================================
# CELL 3: Quick validation of raw simulations
# =============================================================================
# Fully reads only a few simulations (to detect vol_shape
# and verify integrity). The rest is validated only by file size
# (.raw >= expected_bytes), avoiding reading 6879 files of 32 MB each.
# =============================================================================

def validate_raw_sims(raw_sims_list, max_full_read=10):
    """
    Validates raw simulations efficiently.
    - First max_full_read: full read (verify shape, dose>0)
    - Rest: file size verification of .raw only
    """
    valid = []
    skipped = []
    vol_shape = None

    n_total = len(raw_sims_list)
    if n_total == 0:
        return valid, skipped, EXPECTED_VOL_SHAPE

    # ── Full read of the first N ──
    print(f"  Full validation of {min(max_full_read, n_total)} samples...")
    for i, sim in enumerate(raw_sims_list[:max_full_read]):
        sid = sim['sid']
        try:
            dose_arr, dose_meta = load_mhd(sim['dose_path'])

            if vol_shape is None:
                vol_shape = dose_arr.shape
                print(f"  Reference shape: {vol_shape} (from {sid})")

            if dose_arr.shape != vol_shape:
                skipped.append((sid, f"shape {dose_arr.shape} != {vol_shape}"))
                continue

            if dose_arr.max() == 0:
                skipped.append((sid, "max dose = 0 (failed simulation)"))
                continue

            valid.append(sim)
            print(f"  ✓ {sid}: shape={dose_arr.shape}, "
                  f"max={dose_arr.max():.4e} Gy")

        except Exception as e:
            skipped.append((sid, str(e)[:60]))

    if vol_shape is None:
        vol_shape = EXPECTED_VOL_SHAPE
        print(f"  Could not detect shape, using default: {vol_shape}")

    # ── Quick verification of the rest (file size only) ──
    remaining = raw_sims_list[max_full_read:]
    if remaining:
        # Determine .raw dtype: float32 = 4 bytes, float64 = 8 bytes
        # Use a conservative minimum (float32)
        expected_bytes = int(np.prod(vol_shape)) * 4  # 200^3 * 4 = 32 MB
        n_quick_ok = 0
        n_quick_skip = 0

        for sim in remaining:
            sid = sim['sid']
            dose_raw_path = sim['dose_path'].replace('.mhd', '.raw')
            try:
                fsize = os.path.getsize(dose_raw_path)
                if fsize >= expected_bytes:
                    valid.append(sim)
                    n_quick_ok += 1
                else:
                    skipped.append((sid, f".raw file too small: "
                                        f"{fsize/1e6:.1f} MB < "
                                        f"{expected_bytes/1e6:.0f} MB"))
                    n_quick_skip += 1
            except Exception as e:
                skipped.append((sid, str(e)[:60]))
                n_quick_skip += 1

        print(f"  Quick verification ({len(remaining)} remaining): "
              f"{n_quick_ok} OK, {n_quick_skip} saltadas")

    return valid, skipped, vol_shape


# ─── Validate ───
valid_raw = []
skipped_raw = []
detected_vol_shape = EXPECTED_VOL_SHAPE

if raw_sims:
    print(f"Validating {len(raw_sims)} raw simulations...\n")
    valid_raw, skipped_raw, detected_shape = validate_raw_sims(raw_sims)
    if detected_shape:
        detected_vol_shape = detected_shape

    print(f"\n  Valid:     {len(valid_raw)}")
    if skipped_raw:
        print(f"  Skipped:   {len(skipped_raw)}")
        for sid, reason in skipped_raw[:10]:
            print(f"    ✗ {sid}: {reason}")

# Validate partial HDF5
if h5_partials:
    print(f"\nPartial HDF5 ({len(h5_partials)}):")
    for ds in h5_partials:
        if ds['volume_shape'] != EXPECTED_VOL_SHAPE:
            print(f"  ✗ {ds['source']}: vol={ds['volume_shape']} "
                  f"!= expected {EXPECTED_VOL_SHAPE}")
        else:
            print(f"  ✓ {ds['source']}: {ds['n_samples']} samples, OK")

# Total summary of data available for merging
n_total_available = len(valid_raw) + sum(d['n_samples'] for d in h5_partials) + \
                    sum(d['n_samples'] for d in consolidated_prev)
print(f"\nTotal samples available for consolidation: {n_total_available}")
if consolidated_prev:
    n_prev = sum(d['n_samples'] for d in consolidated_prev)
    print(f"  (of which {n_prev} are already in previous consolidated files)")
is_valid = n_total_available > 0

In [ ]:
# =============================================================================
# CELL 4: EBRT parameter space coverage analysis
# =============================================================================
# Extracts parameters from ALL sources (HDF5 + raw) and analyzes
# coverage: unique combinations, realizations, clinical distributions.
# =============================================================================

def extract_params_from_raw(sim):
    """Extracts and normalizes the 10 parameters from a raw simulation."""
    meta = sim['metadata']
    bq = meta.get('beam', {}).get('beam_quality', '6MV')
    beam_info = meta.get('beam', {})
    geom = meta.get('geometry', {})
    field_meta = meta.get('field', {})

    raw_params = {
        'beam_energy_norm':  ENERGY_MAP.get(bq, 0.0),
        'is_fff':            FFF_MAP.get(bq, 0.0),
        'field_x_cm':        field_meta.get('jaw_size_x_cm', 10.0),
        'field_y_cm':        field_meta.get('jaw_size_y_cm', 10.0),
        'ssd_cm':            beam_info.get('ssd_cm', 100.0),
        'gantry_angle':      beam_info.get('gantry_angle_deg', 0.0),
        'collimator_angle':  beam_info.get('collimator_angle_deg', 0.0),
        'patient_type':      PATIENT_TYPE_MAP.get(
                                 meta.get('patient_type', 'pelvis'), 0.0),
        'tumor_depth':       geom.get('tumor_depth_cm', 5.0),
        'tumor_size':        geom.get('tumor_size_cm', 3.0),
    }

    norm_vec = []
    for key in EBRT_PARAM_RANGES:
        lo, hi = EBRT_PARAM_RANGES[key]
        val = raw_params.get(key, lo)
        norm_vec.append((val - lo) / (hi - lo) if hi > lo else 0.0)

    return np.array(norm_vec, dtype=np.float32)


def analyze_coverage(h5_datasets, raw_valid):
    """Analyzes parameter space coverage combining all sources."""
    all_params_list = []
    param_counts = defaultdict(int)

    # Parameters from HDF5
    for ds in h5_datasets:
        with h5py.File(ds['path'], 'r') as hf:
            params = hf['params'][:]
            all_params_list.append(params)

    # Parameters from raw simulations
    if raw_valid:
        raw_params_arr = np.array([extract_params_from_raw(s) for s in raw_valid])
        all_params_list.append(raw_params_arr)

    if not all_params_list:
        return {}, None

    all_params = np.concatenate(all_params_list, axis=0)

    for i in range(all_params.shape[0]):
        key = tuple(np.round(all_params[i], decimals=6))
        param_counts[key] += 1

    counts = list(param_counts.values())
    stats = {
        'n_total': all_params.shape[0],
        'n_unique_combos': len(param_counts),
        'avg_realizations': np.mean(counts),
        'max_realizations': max(counts),
        'min_realizations': min(counts),
    }
    return stats, all_params


def print_ebrt_distributions(all_params):
    """Prints EBRT clinical distributions."""
    n = all_params.shape[0]

    # Patient type (param 7)
    pt = all_params[:, 7]
    n_pelvis = np.sum(np.abs(pt - 0.0) < 0.1)
    n_hn = np.sum(np.abs(pt - 0.5) < 0.1)
    n_lung = np.sum(np.abs(pt - 1.0) < 0.1)
    print(f"\n   Patient type:")
    print(f"     Pelvis:       {n_pelvis:>5} ({100*n_pelvis/n:.1f}%)")
    print(f"     Head & neck:  {n_hn:>5} ({100*n_hn/n:.1f}%)")
    print(f"     Lung:         {n_lung:>5} ({100*n_lung/n:.1f}%)")

    # Beam quality (params 0 and 1)
    energy = all_params[:, 0]
    fff = all_params[:, 1]
    beams = {
        '6MV':   np.sum((np.abs(energy - 0.0) < 0.1) & (fff < 0.5)),
        '6FFF':  np.sum((np.abs(energy - 0.0) < 0.1) & (fff > 0.5)),
        '10MV':  np.sum((np.abs(energy - 0.5) < 0.1) & (fff < 0.5)),
        '10FFF': np.sum((np.abs(energy - 0.5) < 0.1) & (fff > 0.5)),
        '15MV':  np.sum((np.abs(energy - 1.0) < 0.1) & (fff < 0.5)),
    }
    print(f"\n   Beam quality:")
    for label, count in beams.items():
        print(f"     {label:>5s}:  {count:>5} ({100*count/n:.1f}%)")

    # SSD (param 4)
    ssd_norm = all_params[:, 4]
    ssd_cm = ssd_norm * 30.0 + 80.0
    unique_ssd = np.unique(np.round(ssd_cm, 0))
    print(f"\n   SSD ({len(unique_ssd)} valores):")
    for ssd in sorted(unique_ssd):
        count = np.sum(np.abs(ssd_cm - ssd) < 1.0)
        print(f"     {ssd:>5.0f} cm:  {count:>5} ({100*count/n:.1f}%)")


# ─── Analyze ───
coverage_stats = {}
all_params_array = None

if is_valid:
    print("Analyzing EBRT parameter space coverage...\n")
    all_h5 = consolidated_prev + h5_partials
    coverage_stats, all_params_array = analyze_coverage(all_h5, valid_raw)

    if coverage_stats:
        print(f"   Total samples:             {coverage_stats['n_total']}")
        print(f"   Unique combinations:       {coverage_stats['n_unique_combos']}")
        print(f"   Realizations/combination:  "
              f"min={coverage_stats['min_realizations']}, "
              f"mean={coverage_stats['avg_realizations']:.1f}, "
              f"max={coverage_stats['max_realizations']}")

        if all_params_array is not None:
            print_ebrt_distributions(all_params_array)
else:
    print("No data to analyze.")

In [ ]:
# =============================================================================
# CELL 5: Incremental merge — ONLY NEW samples + batch upload
# =============================================================================
# KEY CHANGE: Does NOT copy samples from previous consolidated files.
# Each run produces one or more independent partial files
# with only raw samples that have not been processed before.
#
# BATCH UPLOAD (Kaggle): When the disk fills up (~19 GB), the partial
# HDF5 is uploaded to Kaggle as a dataset, deleted from disk to free
# space, and processing continues with pending samples. This allows
# processing ALL samples in ONE SINGLE run.
#
# STORAGE OPTIMIZATION:
#   - density → float16 (significantly reduces size)
#   - dose → float32 (maintain precision for values ~1e-7 to 1e-4 Gy)
#   - MC uncertainty → NOT STORED (analysis concluded it does not contribute to
#     training: dose-weighted MSE already approximates optimal 1/σ² weighting,
#     savings are ~30% of storage, and MC Dropout is superior for
#     epistemic uncertainty estimation in predictions)
#   - Mean uncertainty per sample is computed as a QUALITY METRIC
#   - The training notebook automatically casts float16→float32 when reading
#
# DISK MONITORING: Kaggle has ~20 GB. Each sample ≈ 12-14 MB compressed
# (previously ~17-19 MB when uncertainty was stored).
#
# DEDUPLICATION:
#   1. _index.json from previous runs (via discover_resume_index)
#   2. source_mapping from previous consolidated files (fallback)
# =============================================================================

MIN_FREE_GB = 1.5  # Stop when less than this remains


def _disk_free_gb(path):
    """Free space in GB."""
    try:
        st = os.statvfs(path)
        return (st.f_bavail * st.f_frsize) / (1024**3)
    except Exception:
        return float('inf')


def _load_consolidated_sids(index_path):
    """Loads the set of SIDs already consolidated from previous runs."""
    sids = set()
    if os.path.exists(index_path):
        try:
            with open(index_path) as f:
                data = json.load(f)
            sids = set(data.get('consolidated_sids', []))
            print(f"  Resume index: {len(sids)} SIDs already consolidated")
        except Exception:
            pass
    return sids


def _extract_sids_from_h5(h5_path):
    """
    Extracts SIDs from the source_mapping attribute of a consolidated HDF5.
    Fallback for when there is no _index.json (e.g.: consolidated with old code).
    """
    sids = set()
    try:
        with h5py.File(h5_path, 'r') as hf:
            mapping_str = hf.attrs.get('source_mapping', '[]')
            if isinstance(mapping_str, (str, bytes)):
                entries = json.loads(mapping_str)
            else:
                entries = []
            for entry in entries:
                if isinstance(entry, dict) and 'sid' in entry:
                    sids.add(entry['sid'])
    except Exception:
        pass
    return sids


def _save_consolidated_index(index_path, sids_set, stats_dict):
    """Saves the index of consolidated SIDs for resume."""
    data = {
        'consolidated_sids': sorted(sids_set),
        'n_consolidated': len(sids_set),
        'stats': stats_dict,
    }
    with open(index_path, 'w') as f:
        json.dump(data, f, indent=2)


def merge_new_samples(consolidated_prev_list, h5_partials_list,
                      raw_valid, output_path, vol_shape):
    """
    Writes ONLY new samples to the output HDF5.

    - Does NOT copy from previous consolidated files (they are already saved as
      separate datasets on Kaggle — it makes no sense to copy 19 GB only to then
      not have space for new ones).
    - Uses _index.json + source_mapping to know which SIDs are already done.
    - Copies non-consolidated partial HDF5 if any exist (rare).
    - density is stored as float16 to save space.
    - MC uncertainty is NOT stored (not useful for training; see analysis
      in red_neuronal_EBRT). Computed and logged as a quality metric.
    """
    index_path = output_path.replace('.h5', '_index.json')

    # ── Build set of already-done SIDs ──
    already_done = _load_consolidated_sids(index_path)

    # Fallback: extract SIDs from previous consolidated files (if no _index.json)
    for ds in consolidated_prev_list:
        prev_sids = _extract_sids_from_h5(ds['path'])
        if prev_sids:
            print(f"  SIDs extracted from {ds['source']}: {len(prev_sids)}")
            already_done |= prev_sids

    n_already = len(already_done)
    if n_already > 0:
        print(f"  Total SIDs ya consolidados: {n_already}")

    # Filter raw: only pending
    raw_pending = [s for s in raw_valid if s['sid'] not in already_done]
    raw_skipped = len(raw_valid) - len(raw_pending)
    if raw_skipped > 0:
        print(f"  Resume: {raw_skipped} raw already consolidated → skipping")
        print(f"  Pending: {len(raw_pending)} new raw simulations")

    # Non-consolidated partial HDF5 (if any, they are copied)
    n_from_partials = sum(d['n_samples'] for d in h5_partials_list)
    n_from_raw = len(raw_pending)
    n_total = n_from_partials + n_from_raw

    if n_total == 0:
        print(f"\n  All consolidated! No new samples to process.")
        n_all = n_already + sum(d['n_samples'] for d in consolidated_prev_list)
        if n_all > 0:
            print(f"  You have {n_all} samples in "
                  f"{len(consolidated_prev_list)} consolidated files.")
            print(f"  Add them all as inputs in red_neuronal_EBRT.ipynb")
        return None

    # Estimate space (without uncertainty → ~12-14 MB/sample)
    n_voxels = np.prod(vol_shape)
    # dose(f32) + density(f16) ≈ (4+2) bytes × gzip-4 compression
    est_per_sample_mb = n_voxels * (4 + 2) / 1e6 * 0.3  # ~12-14 MB gzip-4
    free_gb = _disk_free_gb(os.path.dirname(output_path))
    max_samples_disk = int((free_gb - MIN_FREE_GB) * 1000 / est_per_sample_mb)

    print(f"\nCreating partial EBRT dataset: {output_path}")
    if consolidated_prev_list:
        n_prev = sum(d['n_samples'] for d in consolidated_prev_list)
        print(f"   Previous consolidated: {n_prev} samples "
              f"({len(consolidated_prev_list)} files) — NOT copied")
    if n_from_partials > 0:
        print(f"   Partial HDF5:          {n_from_partials} samples (to copy)")
    print(f"   New raw:               {n_from_raw} samples (to reconstruct)")
    print(f"   Total to write:        {n_total} samples")
    print(f"   Volume shape:          {vol_shape}")
    print(f"   Format:                dose=float32, density=float16 (no uncertainty)")
    print(f"   ~{est_per_sample_mb:.0f} MB/sample compressed")
    print(f"   Free disk:             {free_gb:.1f} GB (min: {MIN_FREE_GB} GB)")
    print(f"   ~{max_samples_disk} samples fit on disk")
    if max_samples_disk < n_total:
        if BATCH_UPLOAD_ENABLED:
            print(f"   BATCH UPLOAD active → batches will be uploaded and freed")
        else:
            print(f"   NOTE: Only ~{max_samples_disk} of {n_total} fit.")
            print(f"         The rest will be processed in the next run.")
    print()

    t0 = time.time()
    global_dose_max = 0.0
    write_idx = 0
    source_list = []
    skipped = []
    new_sids = set()
    disk_full = False

    with h5py.File(output_path, 'w') as hf_out:
        # Create datasets — density as float16 to save space
        # The training notebook automatically casts to float32 when reading
        # NOTE: MC uncertainty is NOT stored (see comment at the top)
        ds_density = hf_out.create_dataset(
            'density', shape=(n_total, 1, *vol_shape),
            dtype='float16', chunks=(1, 1, *vol_shape),
            maxshape=(None, 1, *vol_shape),
            compression='gzip', compression_opts=4)
        ds_params = hf_out.create_dataset(
            'params', shape=(n_total, N_PARAMS_EXPECTED),
            dtype='float32', maxshape=(None, N_PARAMS_EXPECTED))
        ds_dose = hf_out.create_dataset(
            'dose', shape=(n_total, 1, *vol_shape),
            dtype='float32', chunks=(1, 1, *vol_shape),
            maxshape=(None, 1, *vol_shape),
            compression='gzip', compression_opts=4)

        # ─── Part 1: Non-consolidated partial HDF5 (if any) ───
        for ds_info in h5_partials_list:
            if disk_full:
                break
            src_label = ds_info['source']
            with h5py.File(ds_info['path'], 'r') as hf_src:
                n_src = hf_src['density'].shape[0]
                for j in range(n_src):
                    if write_idx % 20 == 0 and write_idx > 0:
                        free = _disk_free_gb(os.path.dirname(output_path))
                        if free < MIN_FREE_GB:
                            print(f"\n   LOW DISK: {free:.2f} GB")
                            disk_full = True
                            break

                    ds_density[write_idx] = np.float16(hf_src['density'][j])
                    ds_params[write_idx] = hf_src['params'][j]
                    ds_dose[write_idx] = hf_src['dose'][j]

                    sample_dmax = float(hf_src['dose'][j].max())
                    global_dose_max = max(global_dose_max, sample_dmax)
                    source_list.append({
                        'source': src_label, 'original_idx': int(j),
                        'type': 'hdf5_partial'
                    })
                    write_idx += 1

                    if write_idx % 50 == 0:
                        elapsed = time.time() - t0
                        rate = write_idx / elapsed if elapsed > 0 else 0
                        free = _disk_free_gb(os.path.dirname(output_path))
                        print(f"   [{write_idx}/{n_total}] {rate:.1f} m/s | "
                              f"disco: {free:.1f} GB")
            if not disk_full:
                print(f"   HDF5 {src_label}: {n_src} samples copied")

        # ─── Part 2: Reconstruct from new raw files ───
        if raw_pending and not disk_full:
            print(f"\n   Reconstructing {len(raw_pending)} new raw samples...")

        for sim in raw_pending:
            if disk_full:
                break

            sid = sim['sid']

            # Check disk every 10 samples
            if write_idx % 10 == 0 and write_idx > 0:
                free = _disk_free_gb(os.path.dirname(output_path))
                if free < MIN_FREE_GB:
                    print(f"\n   LOW DISK: {free:.2f} GB < {MIN_FREE_GB} GB")
                    print(f"   Stopping at [{write_idx}/{n_total}] to "
                          f"close HDF5 cleanly.")
                    disk_full = True
                    break

            try:
                dose_arr, dose_meta = load_mhd(sim['dose_path'])
                if dose_arr.shape != vol_shape:
                    skipped.append((sid, f"shape {dose_arr.shape}"))
                    continue

                density = build_ebrt_density(
                    sim['metadata'], vol_shape,
                    dose_meta['spacing'], dose_meta['offset'])
                density = density / 1.85
                norm_params = extract_params_from_raw(sim)

                ds_density[write_idx, 0, ...] = density.astype(np.float16)
                ds_params[write_idx, :] = norm_params
                ds_dose[write_idx, 0, ...] = dose_arr.astype(np.float32)

                # Compute mean uncertainty as quality metric (without storing)
                mean_uncert = None
                if sim['uncert_path'] and os.path.exists(sim['uncert_path']):
                    try:
                        uncert_arr, _ = load_mhd(sim['uncert_path'])
                        # Mean σ/D in significant dose region (>10% Dmax)
                        dmax_local = dose_arr.max()
                        if dmax_local > 0:
                            high_dose_mask = dose_arr > 0.1 * dmax_local
                            if high_dose_mask.any():
                                mean_uncert = float(uncert_arr[high_dose_mask].mean())
                    except Exception:
                        pass

                sample_dmax = float(dose_arr.max())
                global_dose_max = max(global_dose_max, sample_dmax)
                batch_name = os.path.basename(sim['batch_dir'])
                source_list.append({
                    'source': batch_name, 'sid': sid,
                    'type': 'raw_reconstructed',
                    'mean_uncertainty': mean_uncert,
                })
                new_sids.add(sid)
                write_idx += 1

                if write_idx % 20 == 0 or write_idx == n_total:
                    elapsed = time.time() - t0
                    rate = write_idx / elapsed if elapsed > 0 else 0
                    eta = (n_total - write_idx) / rate if rate > 0 else 0
                    free = _disk_free_gb(os.path.dirname(output_path))
                    bq = sim['metadata'].get('beam', {}).get(
                        'beam_quality', '?')
                    pt = sim['metadata'].get('patient_type', '?')
                    print(f"   [{write_idx}/{n_total}] {sid} | {bq} | {pt}"
                          f" | dmax={sample_dmax:.4e} | {rate:.1f} m/s"
                          f" | disco: {free:.1f} GB")

            except Exception as e:
                skipped.append((sid, str(e)[:80]))
                print(f"   ✗ {sid}: {str(e)[:60]}")

        # ─── Resize to the actual number of written samples ───
        if write_idx < n_total:
            for dsname in ['density', 'params', 'dose']:
                hf_out[dsname].resize(write_idx, axis=0)

        # ─── Global attributes ───
        hf_out.attrs['n_scenarios'] = write_idx
        hf_out.attrs['n_valid'] = write_idx
        hf_out.attrs['volume_shape'] = list(vol_shape)
        hf_out.attrs['global_dose_max'] = float(global_dose_max)
        hf_out.attrs['dose_max_global'] = float(global_dose_max)
        hf_out.attrs['param_names'] = EBRT_PARAM_NAMES
        hf_out.attrs['param_ranges'] = json.dumps(
            {k: list(v) for k, v in EBRT_PARAM_RANGES.items()})
        hf_out.attrs['modality'] = 'EBRT_photon'
        hf_out.attrs['machine'] = 'Varian_TrueBeam_SVC_3.0'
        hf_out.attrs['consolidated'] = True
        hf_out.attrs['is_partial'] = True  # Always partial in incremental mode
        hf_out.attrs['n_from_raw'] = write_idx
        hf_out.attrs['source_mapping'] = json.dumps(source_list[:1000])
        hf_out.attrs['dtype_density'] = 'float16'
        hf_out.attrs['dtype_dose'] = 'float32'
        if coverage_stats:
            hf_out.attrs['n_unique_param_combos'] = coverage_stats.get(
                'n_unique_combos', 0)
            hf_out.attrs['avg_realizations_per_combo'] = float(
                coverage_stats.get('avg_realizations', 0))

    # ─── Update resume index ───
    all_done = already_done | new_sids
    _save_consolidated_index(index_path, all_done, {
        'write_idx': write_idx,
        'n_total_raw': len(raw_valid),
        'n_pending_after': len(raw_valid) - len(all_done),
        'disk_full': disk_full,
        'global_dose_max': float(global_dose_max),
    })

    elapsed_total = time.time() - t0
    file_size_mb = os.path.getsize(output_path) / 1e6

    print(f"\nPartial EBRT dataset created")
    print(f"   Path:             {output_path}")
    print(f"   New samples:      {write_idx}")
    print(f"   Dose max:         {global_dose_max:.6e} Gy")
    print(f"   Size:             {file_size_mb:.1f} MB ({file_size_mb/1e3:.2f} GB)")
    print(f"   Format:           density=float16, dose=float32 (no uncertainty)")
    print(f"   Time:             {elapsed_total:.1f}s ({elapsed_total/60:.1f} min)")
    if skipped:
        print(f"   Skipped:          {len(skipped)}")
        for sid, reason in skipped[:5]:
            print(f"     ✗ {sid}: {reason}")

    # ── Global progress ──
    total_done = len(all_done)
    total_raw = len(raw_valid)
    remaining = total_raw - total_done
    n_prev = sum(d['n_samples'] for d in consolidated_prev_list)

    print(f"\n   {'='*60}")
    print(f"   INCREMENTAL PROGRESS")
    print(f"   In previous consolidated: {n_prev} samples")
    print(f"   In this file:             {write_idx} new samples")
    print(f"   Total consolidadas:      {total_done}/{total_raw} "
          f"({100*total_done/total_raw:.1f}%)" if total_raw > 0 else "")

    if remaining > 0:
        runs_est = int(np.ceil(remaining / max(write_idx, 1)))
        print(f"   Pending:                  {remaining}")
        if BATCH_UPLOAD_ENABLED:
            print(f"   Batch upload active → will continue automatically")
        else:
            print(f"   Additional runs:          ~{runs_est}")
        print(f"   {'='*60}")
    else:
        print(f"   {'='*60}")
        print(f"   CONSOLIDATION COMPLETE!")
        print(f"   Partial files: {len(consolidated_prev_list) + 1}")
        print(f"   Add ALL as inputs in red_neuronal_EBRT.ipynb")

    return output_path, remaining, all_done


# =============================================================================
# Run merge with automatic batch upload
# =============================================================================
if is_valid:
    part_number = _get_next_part_number()
    total_written_session = 0
    total_uploaded_session = 0
    t_session = time.time()

    while True:
        ret = merge_new_samples(
            consolidated_prev, h5_partials, valid_raw,
            OUTPUT_H5, detected_vol_shape)

        if ret is None:
            # Nothing to process (everything already consolidated)
            break

        output_path, remaining, all_done_sids = ret
        total_written_session += os.path.getsize(output_path) / 1e6 \
            if os.path.exists(output_path) else 0

        if remaining == 0:
            # All processed — the last file remains on disk as output
            print(f"\n   All samples processed in this session!")
            break

        if not BATCH_UPLOAD_ENABLED:
            # Without batch upload — original behavior
            print(f"\n   NEXT STEP:")
            print(f"   1. Save this output as a Kaggle Dataset")
            print(f"   2. Add it as Input along with:")
            print(f"      - All simulation outputs (raw)")
            print(f"      - All previous consolidated files")
            print(f"   3. Run again → will process {remaining} remaining")
            print(f"   IMPORTANT: DO NOT remove the simulation outputs from "
                  f"the inputs")
            break

        # ─── Batch upload: upload and free disk ───
        print(f"\n   {remaining} samples remaining → uploading batch and continuing...")

        success = _upload_kaggle_dataset(OUTPUT_H5, part_number)
        if not success:
            print(f"\n   Upload failed. The file remains on disk.")
            print(f"   Save it manually as a Kaggle Dataset.")
            break

        total_uploaded_session += 1
        part_number += 1

        # Release old index so merge_new_samples re-reads it
        # (the index was already updated in merge_new_samples, no need to touch it)
        print(f"\n   Disk freed → continuing with {remaining} pending...\n")

    # ─── Session summary ───
    session_time = time.time() - t_session
    print(f"\n{'='*65}")
    print(f"   SESSION SUMMARY")
    print(f"   Total time:       {session_time:.0f}s ({session_time/60:.1f} min)")
    if total_uploaded_session > 0:
        print(f"   Batches uploaded: {total_uploaded_session}")
        print(f"   Current part:     local file at {OUTPUT_H5}")
        print(f"\n   For training, add as inputs in red_neuronal_EBRT.ipynb:")
        print(f"   - All 'ebrt-consolidated-part-NNN' (uploaded)")
        print(f"   - The output of THIS notebook (last batch)")
    print(f"{'='*65}")

    # result for the quality cell
    result = OUTPUT_H5 if os.path.exists(OUTPUT_H5) else None
else:
    result = None
    print("Cannot merge: no valid data found.")

In [ ]:
# =============================================================================
# CELL 6: Quality report of the consolidated EBRT dataset
# =============================================================================

def quality_report(h5_path):
    """Generates a quality report of the consolidated EBRT dataset."""
    with h5py.File(h5_path, 'r') as hf:
        n = hf['density'].shape[0]
        vol_shape = tuple(hf['density'].shape[2:])

        print("=" * 65)
        print("    QUALITY REPORT — Consolidated EBRT Dataset")
        print("    Varian TrueBeam SVC 3.0 + HD 120 MLC")
        print("=" * 65)

        # Structure
        print(f"\nHDF5 Structure:")
        for key in hf.keys():
            ds = hf[key]
            print(f"   /{key:20s} shape={ds.shape}  dtype={ds.dtype}")

        # Attributes
        print(f"\nAttributes:")
        for key, val in sorted(hf.attrs.items()):
            if key in ('source_mapping',):
                try:
                    parsed = json.loads(val)
                    print(f"   {key:30s} = [{len(parsed)} entries]")
                except Exception:
                    print(f"   {key:30s} = (data)")
                continue
            val_str = str(val)
            if len(val_str) > 80:
                val_str = val_str[:77] + '...'
            print(f"   {key:30s} = {val_str}")

        # Parameter statistics
        params = hf['params'][:]
        print(f"\nParameter distribution ({n} samples):")
        print(f"   {'Parameter':25s} {'min':>8s} {'max':>8s} {'mean':>8s} "
              f"{'std':>8s}  real range")
        print(f"   {'─'*85}")
        for j, pname in enumerate(EBRT_PARAM_NAMES):
            col = params[:, j]
            real_range = ''
            if pname in EBRT_PARAM_RANGES:
                pmin, pmax = EBRT_PARAM_RANGES[pname]
                real_min = col.min() * (pmax - pmin) + pmin
                real_max = col.max() * (pmax - pmin) + pmin
                real_range = f'[{real_min:.1f}, {real_max:.1f}]'
            print(f"   {pname:25s} {col.min():8.4f} {col.max():8.4f} "
                  f"{col.mean():8.4f} {col.std():8.4f}  {real_range}")

        # EBRT distributions
        pt = params[:, 7]
        n_pelvis = np.sum(np.abs(pt - 0.0) < 0.1)
        n_hn = np.sum(np.abs(pt - 0.5) < 0.1)
        n_lung = np.sum(np.abs(pt - 1.0) < 0.1)
        energy = params[:, 0]
        fff = params[:, 1]
        beams = {
            '6MV':   np.sum((np.abs(energy - 0.0) < 0.1) & (fff < 0.5)),
            '6FFF':  np.sum((np.abs(energy - 0.0) < 0.1) & (fff > 0.5)),
            '10MV':  np.sum((np.abs(energy - 0.5) < 0.1) & (fff < 0.5)),
            '10FFF': np.sum((np.abs(energy - 0.5) < 0.1) & (fff > 0.5)),
            '15MV':  np.sum((np.abs(energy - 1.0) < 0.1) & (fff < 0.5)),
        }
        print(f"\nClinical distributions:")
        print(f"   Patients: pelvis={n_pelvis}, head_neck={n_hn}, lung={n_lung}")
        beam_str = ', '.join(f'{k}={v}' for k, v in beams.items())
        print(f"   Beam:      {beam_str}")

        # Coverage
        n_unique = int(hf.attrs.get('n_unique_param_combos', 0))
        avg_real = float(hf.attrs.get('avg_realizations_per_combo', 0))
        if n_unique > 0:
            print(f"\nMonte Carlo Coverage:")
            print(f"   Unique combinations: {n_unique}")
            print(f"   Realizations/combo:  ~{avg_real:.1f}")

        # Dose statistics (sampling)
        n_sample = min(n, 100)
        print(f"\nDose statistics (sampling {n_sample}):")
        dose_maxs = []
        dose_zeros = 0
        sample_indices = np.linspace(0, n-1, n_sample, dtype=int)
        for idx in sample_indices:
            dose = hf['dose'][idx]
            dmax = float(dose.max())
            dose_maxs.append(dmax)
            if dmax == 0:
                dose_zeros += 1
        print(f"   Dose max:  min={min(dose_maxs):.4e}, max={max(dose_maxs):.4e}, "
              f"mean={np.mean(dose_maxs):.4e}")
        if dose_zeros > 0:
            print(f"   WARNING: {dose_zeros}/{n_sample} samples with dose=0")

        # Density
        print(f"\nDensity statistics (sampling):")
        dens_maxs = []
        for idx in sample_indices:
            d = np.float32(hf['density'][idx])  # Convert float16→float32 for stats
            dens_maxs.append(float(d.max()))
        if max(dens_maxs) > 1.1:
            print(f"   WARNING: densities > 1.1 — normalization ÷1.85?")
        else:
            print(f"   Normalization OK: max ~{max(dens_maxs):.2f} "
                  f"(hueso/1.85 = 1.0)")

        # Origin
        n_hdf5 = int(hf.attrs.get('n_from_hdf5', 0))
        n_raw = int(hf.attrs.get('n_from_raw', 0))
        print(f"\nSample origin:")
        print(f"   From HDF5:           {n_hdf5}")
        print(f"   Reconstructed (raw): {n_raw}")

        # Storage format
        dtype_dens = str(hf.attrs.get('dtype_density', hf['density'].dtype))
        dtype_dose = str(hf.attrs.get('dtype_dose', hf['dose'].dtype))
        dtype_unc = str(hf.attrs.get('dtype_uncertainty', hf['uncertainty'].dtype)) \
            if 'uncertainty' in hf else 'N/A'
        print(f"\nStorage format:")
        print(f"   density:      {dtype_dens}")
        print(f"   dose:         {dtype_dose}")
        print(f"   uncertainty:  {dtype_unc}")

        # Final summary
        file_size = os.path.getsize(h5_path) / 1e6
        print(f"\n{'='*65}")
        print(f"   File:       {file_size:.1f} MB ({file_size/1e3:.2f} GB)")
        print(f"   {n} muestras x vol{vol_shape} x 10 params")
        print(f"   Machine:    Varian TrueBeam SVC 3.0 + HD 120 MLC")
        print(f"   Qualities:  6MV, 6FFF, 10MV, 10FFF, 15MV")
        print(f"   Patients:   pelvis, head_neck, lung")
        print("=" * 65)


# ─── Generate report ───
if result and os.path.exists(result):
    quality_report(result)
elif result:
    print(f"The file {result} was uploaded to Kaggle and deleted from disk.")
    print("The quality report ran during the batch upload process.")
else:
    print("No dataset to analyze.")

---

## Next step

### With batch upload (automatic on Kaggle)

The notebook uploads each partial batch (~810-1200 samples) to Kaggle as an independent dataset (`ebrt-consolidated-part-NNN`), frees disk space, and continues processing. **All samples are processed in ONE single run.**

1. **Run** all cells → batches are uploaded automatically
2. The **last batch** remains as the notebook output → **save it as a Kaggle Dataset**
3. In `red_neuronal_EBRT.ipynb`, add as **Input**:
   - All `ebrt-consolidated-part-NNN` (uploaded via API)
   - The output of this notebook (last batch)

### Without batch upload (manual mode)

If the upload fails or is unavailable:

1. **Save the output** of this notebook as a **Kaggle Dataset**
2. Add it as Input along with the simulation outputs
3. Run again → will process pending samples (automatic resume)

### Storage optimizations

- `density` and `uncertainty` → **float16** (reduces ~30% size per sample)
- `dose` → **float32** (maintains precision for values ~1e-7 to 1e-4 Gy)
- The training notebook automatically casts float16→float32 when reading

### Compatibility

| Notebook | Produces | Consumes |
|---|---|---|
| `ebrt_montecarlo_simulacion.ipynb` | `ebrt_batch/` (raw) + `ebrt_training_dataset.h5` | — |
| `consolidar_datasets_EBRT.ipynb` (this) | `ebrt_consolidated_dataset.h5` + `ebrt-consolidated-part-NNN` (API) | Valid HDF5 + raw `.mhd` (if HDF5 corrupt) |
| `red_neuronal_EBRT.ipynb` | `checkpoint_ebrt.pth` | All consolidated HDF5 files (loads multiple files) |

---